In [22]:
import numpy as np
import pyarrow.dataset as ds

def compute_feature_stats(
    paths,
    features,
    cat_cols,
    fold_col=None,
    include_folds=None,
    exclude_folds=None,
    batch_rows=200_000,
):
    num_cols = [c for c in features if c not in cat_cols]
    dataset = ds.dataset(paths, format="parquet")
    fexpr = None
    if fold_col:
        col = ds.field(fold_col)
        if include_folds is not None:
            fexpr = col.isin(sorted(include_folds))
        if exclude_folds is not None:
            ex = ~col.isin(sorted(exclude_folds))
            fexpr = ex if fexpr is None else (fexpr & ex)

    scanner = dataset.scanner(columns=num_cols, filter=fexpr, batch_size=batch_rows)

    n = 0
    s = None
    s2 = None
    for rb in scanner.to_reader():
        X = rb.to_pandas()[num_cols].to_numpy(dtype=np.float64, copy=False)  # float64で集計
        if s is None:
            d = X.shape[1]
            s = np.zeros(d, dtype=np.float64)
            s2 = np.zeros(d, dtype=np.float64)
        s += X.sum(axis=0)
        s2 += (X * X).sum(axis=0)
        n += X.shape[0]

    mean = s / max(n, 1)
    var = s2 / max(n, 1) - mean**2
    var[var < 0] = 0.0  # 数値誤差
    std = np.sqrt(var)
    std[std == 0] = 1.0  # 定数列保護
    return mean.astype(np.float32), std.astype(np.float32)


In [ ]:
from dataclasses import dataclass, field
import torch
import numpy as np

@dataclass(eq=False)
class ParquetStream(IterableDataset):
    ...
    # 追加：標準化オプション
    standardize: bool = False
    mean: np.ndarray | None = None
    std:  np.ndarray | None = None
    norm_indices: np.ndarray | None = None  # 標準化対象の列インデックス（数値列のみ）

    def __post_init__(self):
        super().__init__()
        ...
        # 標準化対象の列インデックス（cat_colsを除く）
        if self.standardize:
            cat_set = set(self.cat_cols or [])
            self._num_feats = [c for c in self.features if c not in cat_set]
            # self.features の中で、数値列の位置
            idx = [self.features.index(c) for c in self._num_feats]
            self.norm_indices = np.asarray(idx, dtype=np.int64)

            assert self.mean is not None and self.std is not None, \
                "standardize=True のときは mean/std を渡してください"
            # mean/std を features 順に並べ替える
            # ここでは self.mean/self.std は _num_feats 順で渡す前提
            self._mean = torch.tensor(self.mean, dtype=torch.float32)
            self._std  = torch.tensor(self.std,  dtype=torch.float32)

    def __iter__(self):
        ...
            for i0 in range(0, len(yb), self.batch_size):
                i1 = min(i0 + self.batch_size, len(yb))
                if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                    return
                xb = torch.from_numpy(Xb[i0:i1]).float()  # (B, F)
                if self.standardize:
                    # 対象列だけ正規化（固定統計）
                    idx = self.norm_indices
                    xb[:, idx] = (xb[:, idx] - self._mean) / self._std

                ybt = torch.from_numpy(yb[i0:i1]).float()
                if wb is None:
                    yield xb, ybt
                else:
                    yield xb, ybt, torch.from_numpy(wb[i0:i1]).float()
                emitted += (i1 - i0)
        ...

In [42]:
rename_dict = {
    "cb_v1": "cb-001",
    "cb_v2": "cb-012",
    "lgbm_v1": "lgbm-001",
    "lgbm_v2": "lgbm-012",
    "lgbm_v3": "lgbm-023",
    "logreg_v1": "logreg-009",
    "logreg_v2": "logreg-017",
    "logreg_v3": "logreg-020",
    "mlp_v1": "mlp-002",
    "mlp_v2": "mlp-004",
    "mlp_v3": "mlp-005",
    "mlp_v4": "mlp-006",
    "mlp_v5": "mlp-007",
    "mlp_v6": "mlp-008",
    "mlp_v7": "mlp-010",
    "mlp_v8": "mlp-020",
    "mlp_v9": "mlp-024",
    "rfc_v1": "rfc-001",
    "rfc_v2": "rfc-012",
    "rfc_v3": "rfc-019",
    "rfc_v4": "rfc-021",
    "xgb_v1": "xgb-001",
    "xgb_v2": "xgb-003",
    "xgb_v3": "xgb-011",
    "xgb_v4": "xgb-012",
    "xgb_v5": "xgb-015",
    "xgb_v6": "xgb-013",
    "xgb_v7": "xgb-016",
    "xgb_v8": "xgb-019",
    "xgb_v9": "xgb-021",
}
# bulk_clone_studies(rename_dict, url)

In [43]:
for study in rename_dict.keys():
    optuna.delete_study(
        study_name=study, storage=url)

In [ ]:
import pyarrow.parquet as pq
import polars as pl

pf = pq.ParquetFile("data.parquet")
schema = pf.schema_arrow
for field in schema:
    if pa.types.is_dictionary(field.type):
        # dictionary のキー(=ユニーク値)は field.dictionary を読む
        dict_type = field.type
        print(field.name, dict_type.dictionary_length)  # ←ユニーク数

In [58]:
import pyarrow.parquet as pq
import polars as pl

data = pl.DataFrame(
    {
        "x": [1, 2, 3],
        "y": [4, 5, 6]
    }
)
data = data.with_columns(
    pl.col("x").cast(pl.Utf8).cast(pl.Categorical)
)
data.write_parquet("data.parquet")

In [59]:
p = pq.ParquetFile("data.parquet")

ArrowInvalid: Unrecognized type: 24

In [17]:
import pyarrow.parquet as pq
import polars as pl

data = pl.DataFrame(
    {
        "x": [1, 2, 3],
        "y": [4, 5, 6]
    }
)
data = data.with_columns(
    pl.col("x").cast(pl.Utf8).cast(pl.Categorical)
)
data.write_parquet("data2.parquet")

In [18]:
data = pl.read_parquet(["data.parquet", "data2.parquet"])

In [28]:
n_rows = (
    pl.scan_parquet(["data.parquet", "data2.parquet"])               # ← list[str]でもOK
      .select(pl.len())                             # 総行数を数える（列は不要）
      .collect(engine="streaming")                      # 大量ファイルでも軽量に
      .item()                                       # 1x1 -> Pythonのintへ
)

In [49]:
import numpy as np
y = pl.read_parquet("data2.parquet", columns=["y"]).get_column("y").cast(pl.Float32).to_numpy()

In [50]:
print(y, type(y))

[4. 5. 6.] <class 'numpy.ndarray'>


In [10]:
def cardinalities(path: str, cat_cols: list[str]) -> dict[str, int]:
    scan = pl.scan_parquet(path)
    return {c: scan.select(pl.col(c).n_unique()).collect().item() for c in cat_cols}

cardinalities("data2.parquet", "x")

{'x': 3}

In [8]:
data.schema

Schema([('x', Categorical), ('y', Int64)])

In [10]:
hdr = pl.read_parquet("data.parquet", n_rows=0)

In [12]:
def cardinalities(path: str, cat_cols: list[str]) -> dict[str, int]:
    scan = pl.scan_parquet(path)
    return {c: scan.select(pl.col(c).n_unique()).collect().item() for c in cat_cols}

In [14]:
cardinalities("data.parquet", "x")

{'x': 1}

In [18]:
from pathlib import Path
path = Path("abc/data.parquet")
print(path, path.resolve())

abc/data.parquet /home/hanse/kaggle/binary-bank/notebooks/abc/data.parquet


# ParquetIterator DEBUG

In [5]:
from torch.utils.data import get_worker_info
print(get_worker_info())

None


In [4]:
print(get_worker_info())

None


In [ ]:
# === CONFIG (you edit here) ===
load_dotenv("../../.env")

@dataclass
class Config:
    # Run
    study_name: str = "mlp-033"
    seed: int = 42

    # Data / CV
    model_name: str = "mlp"
    data_id: str = "033"
    n_fold: int = 5
    fold_idx: int = 0

    # Optuna
    n_trials: int = 30
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

cfg = Config()

# 変更したい追加パラメータ（モデル用の opts）
opts = {
    "max_epochs": cfg.max_epochs,
    "min_epochs": cfg.min_epochs,
    "early_stopping_rounds": cfg.early_stopping_rounds,
}

# basic validation
assert cfg.n_fold >= 2
assert cfg.direction in {"maximize","minimize"}

# W&B
api_key = assert_env("WANDB_API_KEY")
wandb.login(key=api_key)

# 見える化（確認用）
show_cfg(cfg)
print("opts:", json.dumps(opts, indent=2))

In [ ]:
# === Build & Run (frozen) ===
# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")

def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")

# W&B Run（グループ・タグ揃え）
run = wandb.init(
    project=cfg.project,
    group=cfg.group,
    job_type="optuna-search",
    config=dc.asdict(cfg),
    reinit=True,
)

create_objective = get_objective(cfg.model_name)
objective = create_objective(
    cfg.data_id,
    seed=cfg.seed,
    n_fold=cfg.n_fold,
    fold_idx=cfg.fold_idx,
    wandb_project=cfg.project,
    study_name=cfg.study_name,
    opts=opts,  # ← ここだけでモデル側に渡す追加パラメータが完結
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    n_jobs=1,
    direction=cfg.direction,
    study_name=cfg.study_name,
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
)

run.finish()
